# Triton Kernel 主线 · 第 1/10 课：Program、Block 与 Mask

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：建立 Triton program instance 的 SPMD 心智模型并实现任意长度 fill。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、PyTorch 张量、CUDA 基本线程/内存概念
- 本课在路线中的作用：Triton 的 program 类似一个向量化 CTA：一个 program 同时处理 BLOCK 个 lane，而不是显式写单线程代码。

## 核心心智模型

### 1. 它是什么，解决什么问题

Triton 的 program 类似一个向量化 CTA：一个 program 同时处理 BLOCK 个 lane，而不是显式写单线程代码。

### 2. 它如何工作

`program_id` 选择块，`arange` 生成 lane，mask 保护最后一个非完整块；grid 决定 program 数。

### 3. 正确性条件与常见误区

所有可能越界的 load/store 都要 mask；运行时标量不应无故设为 constexpr。

### 4. 性能与工程取舍

BLOCK 越大减少 program 数但提高寄存器/编译压力；先正确再 autotune。

## 具体演示

n=10000、BLOCK=256 时 grid=40，最后 program 只有 16 个有效 lane。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 program 的向量下标。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def fill_kernel(out, n: tl.constexpr, value, BLOCK: tl.constexpr):
    offsets = ______  # TODO: program 起点 + lane
    mask = offsets < n
    tl.store(out + offsets, value, mask=mask)

def fill(n, value):
    out = torch.empty((n,), device="cuda", dtype=torch.float32)
    fill_kernel[(triton.cdiv(n, 256),)](out, n, value, BLOCK=256)
    return out

for n in (1, 257, 10000):
    got = fill(n, 3.25)
    torch.testing.assert_close(got, torch.full_like(got, 3.25))


### 检查方法

在 CUDA/Triton 环境运行本单元格；断言覆盖规则尺寸和非规则尾块。首次 JIT 不计入性能。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“Program、Block 与 Mask”的工作机制。

**你的答案：**


### Q2

删除 store mask 会在哪个 program 越界？

**你的答案：**


### Q3

value 改成 `tl.constexpr` 会怎样影响编译缓存？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def fill_kernel(out, n: tl.constexpr, value, BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < n
    tl.store(out + offsets, value, mask=mask)

def fill(n, value):
    out = torch.empty((n,), device="cuda", dtype=torch.float32)
    fill_kernel[(triton.cdiv(n, 256),)](out, n, value, BLOCK=256)
    return out

for n in (1, 257, 10000):
    got = fill(n, 3.25)
    torch.testing.assert_close(got, torch.full_like(got, 3.25))


### Q1 参考答案

`program_id` 选择块，`arange` 生成 lane，mask 保护最后一个非完整块；grid 决定 program 数。

### Q2 参考答案

判断时先检查本课不变量：所有可能越界的 load/store 都要 mask；运行时标量不应无故设为 constexpr。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：BLOCK 越大减少 program 数但提高寄存器/编译压力；先正确再 autotune。

## 参考资料

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [Triton language API](https://triton-lang.org/main/python-api/triton.language.html)

资料用于建立事实基线；面试回答仍需用自己的语言组织。